In [1]:
import pandas as pd
import umap
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score  #ClaudeCode Ver. 1: needed for systematic UMAP dimension selection below

/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Load the datasets
print("Loading datasets...")
# Replace 'sample_id' with the actual name of the sample column in your methylation data if different
clin_df = pd.read_csv('../data/processed/clinical_kirc_train.csv')

Loading datasets...


In [3]:
clin_df

,Unnamed: 0,sample,tissue_type.samples
0,5,TCGA-DV-5573-01A,Tumor
1,8,TCGA-BP-4795-01A,Tumor
2,9,TCGA-BP-4795-11A,Normal
3,26,TCGA-DV-5565-01A,Tumor
4,27,TCGA-CJ-4869-11A,Normal
...,...,...,...
280,940,TCGA-A3-3385-11A,Normal
281,941,TCGA-A3-3385-01A,Tumor
282,942,TCGA-B8-5552-01B,Tumor
283,946,TCGA-B0-4821-01A,Tumor


In [4]:
meth_df = pd.read_parquet('../data/processed/methylation_kirc_train.parquet')
meth_df

Composite Element REF,cg00000108,cg00000236,cg00000292,cg00000321,cg00000363,cg00000622,cg00000658,cg00000714,cg00000721,cg00000734,...,ctl_69667417,ctl_70664314,ctl_70700334,ctl_71670310,ctl_71678368,ctl_71718498,ctl_73635489,ctl_73784382,ctl_73794434,ctl_74666473
index,,,,,,,,,,,,,,,,,,,,,
TCGA-B0-5108-01A,0.972311,0.909915,0.601544,0.283628,0.221306,0.008769,0.857267,0.132064,0.942689,0.064539,...,0.921408,0.074574,0.067612,0.954321,0.057296,0.196588,0.061329,0.943881,0.950757,0.948703
TCGA-B0-5703-01A,0.964033,0.874732,0.590178,0.454810,0.209447,0.013945,0.931355,0.189218,0.943166,0.063036,...,0.815832,0.058441,0.144345,0.932150,0.056047,0.163352,0.077233,0.903676,0.916601,0.964794
TCGA-CJ-4905-11A,0.960496,0.907699,0.489555,0.437536,0.160331,0.007906,0.914621,0.193818,0.957598,0.061109,...,0.912401,0.043139,0.053916,0.964921,0.091277,0.189204,0.080636,0.940926,0.950549,0.958856
TCGA-B0-4714-01A,0.959082,0.892050,0.539843,0.709672,0.251050,0.010895,0.853589,0.116101,0.945019,0.069490,...,0.914239,0.048749,0.074723,0.960944,0.065973,0.224263,0.061164,0.936791,0.928389,0.939751
TCGA-B2-5635-01A,0.971277,0.933855,0.387262,0.299354,0.242985,0.014074,0.884547,0.176327,0.962483,0.081663,...,0.924102,0.026477,0.071569,0.965354,0.055488,0.140422,0.065160,0.939127,0.953792,0.961929
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-BP-4993-01A,0.965083,0.926505,0.766406,0.518425,0.185070,0.010854,0.831226,0.119747,0.958758,0.043677,...,0.940402,0.015461,0.051947,0.953817,0.047797,0.175196,0.049526,0.955170,0.952150,0.938029
TCGA-CZ-4865-01A,0.969403,0.915849,0.515882,0.515539,0.205803,0.009411,0.901285,0.141882,0.968379,0.055377,...,0.959454,0.018266,0.076597,0.968557,0.044430,0.154470,0.047228,0.961375,0.963338,0.960697
TCGA-BP-4782-11A,0.959676,0.936125,0.496571,0.396565,0.258784,0.009743,0.836323,0.200536,0.950403,0.067827,...,0.913990,0.028057,0.060892,0.967519,0.053769,0.205141,0.057893,0.949411,0.947987,0.945548


In [5]:
# Extract sample IDs from the index
meth_samples = meth_df.index.values

# Extract the raw methylation features (all columns)
meth_features = meth_df.values

# (Optional but highly recommended) Scale the data before UMAP and Clustering
print("Scaling data...")
scaler = StandardScaler()
meth_features_scaled = scaler.fit_transform(meth_features)

#ClaudeCode Ver. 1: systematic UMAP dimensionality selection via silhouette score, replacing the
#hardcoded optimal_n=50. UMAP has no "explained variance" concept like PCA's elbow plot, so we sweep a
#small grid of n_components and pick by silhouette score on a downstream KMeans(k=2) clustering -- a
#label-free criterion, consistent with treating this as a genuinely unsupervised pipeline.
print("Searching for optimal UMAP n_components via silhouette score...")
candidate_n_components = [5, 15, 30]
best_silhouette = -1
optimal_n = candidate_n_components[0]
meth_umap_opt = None
for n in candidate_n_components:
    reducer_search = umap.UMAP(n_components=n, random_state=42)
    embedding_search = reducer_search.fit_transform(meth_features_scaled)
    labels_search = KMeans(n_clusters=2, random_state=42).fit_predict(embedding_search)
    score = silhouette_score(embedding_search, labels_search)
    print(f"  n_components={n}: silhouette={score:.4f}")
    if score > best_silhouette:
        best_silhouette = score
        optimal_n = n
        meth_umap_opt = embedding_search
        reducer_3 = reducer_search
print(f"Selected optimal_n={optimal_n} (silhouette={best_silhouette:.4f})")

# 2. Dimensionality Reduction (UMAP from 406601 to 2)
print("Running UMAP (n=2)...")
# Note: Using the scaled features here, but you can change to meth_features if you prefer raw
reducer_1 = umap.UMAP(n_components=2, random_state=42)
meth_umap_2d = reducer_1.fit_transform(meth_features_scaled)

print("Running UMAP (n=3)...")
# Note: Using the scaled features here, but you can change to meth_features if you prefer raw
reducer_2 = umap.UMAP(n_components=3, random_state=42)
meth_umap_3d = reducer_2.fit_transform(meth_features_scaled)

# 3. Clustering 

N_CLUSTERS_2 = 2

# Feature 1: K-means (K=6) on UMAP data
print("Running K-Means on UMAP 2d data...")
kmeans_umap_2d = KMeans(n_clusters=N_CLUSTERS_2, random_state=42)
kmeans_umap_2d_labels_k2 = kmeans_umap_2d.fit_predict(meth_umap_2d)

# Feature 2: K-means (K=6) on Raw data
print("Running K-Means on Raw data (This may take a while)...")
kmeans_raw = KMeans(n_clusters=N_CLUSTERS_2, random_state=42)
kmeans_raw_labels_k2 = kmeans_raw.fit_predict(meth_features_scaled)

# Feature 1: K-means (K=6) on UMAP data
print("Running K-Means on UMAP 3d data...")
kmeans_umap_3d = KMeans(n_clusters=N_CLUSTERS_2, random_state=42)
kmeans_umap_3d_labels_k2 = kmeans_umap_3d.fit_predict(meth_umap_3d)

#feature 3: K-means on UMAP optimal n data
print(f"Running K-Means on UMAP optimal ({optimal_n}d) data...")
kmeans_umap_opt = KMeans(n_clusters=N_CLUSTERS_2, random_state=42)
kmeans_umap_opt_labels_k2 = kmeans_umap_opt.fit_predict(meth_umap_opt)

# Feature 5: Hierarchical (Agglomerative) on UMAP data
print(f"Running Hierarchical Clustering (K={N_CLUSTERS_2}) on UMAP 2d data...")
# 'ward' linkage minimizes the variance of the clusters being merged
hierarchical_umap_2d = AgglomerativeClustering(n_clusters=N_CLUSTERS_2, linkage='ward')
hierarchical_umap_2d_labels_k2 = hierarchical_umap_2d.fit_predict(meth_umap_2d)

# Feature 6: Hierarchical (Agglomerative) on UMAP 3d data
print(f"Running Hierarchical Clustering (K={N_CLUSTERS_2}) on UMAP 3d data...")
hierarchical_umap_3d = AgglomerativeClustering(n_clusters=N_CLUSTERS_2, linkage='ward')
hierarchical_umap_3d_labels_k2 = hierarchical_umap_3d.fit_predict(meth_umap_3d)

# Feature 7: Hierarchical on UMAP optimal n data
print(f"Running Hierarchical Clustering (K={N_CLUSTERS_2}) on UMAP optimal ({optimal_n}d) data...")
hierarchical_umap_opt = AgglomerativeClustering(n_clusters=N_CLUSTERS_2, linkage='ward')
hierarchical_umap_opt_labels_k2 = hierarchical_umap_opt.fit_predict(meth_umap_opt)

# Feature 8: Hierarchical (Agglomerative) on Raw data
print(f"Running Hierarchical Clustering (K={N_CLUSTERS_2}) on Raw data...")
hierarchical_raw = AgglomerativeClustering(n_clusters=N_CLUSTERS_2, linkage='ward')
hierarchical_raw_labels_k2 = hierarchical_raw.fit_predict(meth_features_scaled)

#Feature 9: GMM on UMAP 2d data
print(f"Running GMM (n_components={N_CLUSTERS_2}) on UMAP 2d data...")
gmm_umap_2d = GaussianMixture(n_components=N_CLUSTERS_2, covariance_type='full',random_state=42, n_init=20)
gmm_umap_2d_labels_k2 = gmm_umap_2d.fit_predict(meth_umap_2d)

# Feature 10: GMM on UMAP 3d data
print(f"Running GMM (n_components={N_CLUSTERS_2}) on UMAP 3d data...")
gmm_umap_3d = GaussianMixture(n_components=N_CLUSTERS_2, covariance_type='full', random_state=42, n_init=20)
gmm_umap_3d_labels_k2 = gmm_umap_3d.fit_predict(meth_umap_3d)

# Feature 11: GMM on UMAP optimal n data
#ClaudeCode Ver. 1: use covariance_type='diag' here, same reasoning as the PCA-optimal-n fix in
#02-1_clustering_pca.ipynb -- a full covariance matrix in optimal_n dimensions needs n*(n+1)/2
#parameters per component (e.g. 465 for n=30) from only ~285 training samples, which overfits and can
#fail to generalize to new data (confirmed via a holdout collapse for the analogous PCA case).
print(f"Running GMM (n_components={N_CLUSTERS_2}) on UMAP optimal ({optimal_n}d) data...")
gmm_umap_opt = GaussianMixture(n_components=N_CLUSTERS_2, covariance_type='diag', random_state=42, n_init=20)
gmm_umap_opt_labels_k2 = gmm_umap_opt.fit_predict(meth_umap_opt)


Scaling data...


Searching for optimal UMAP n_components via silhouette score...


/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


  n_components=5: silhouette=0.5745


...

Running K-Means on UMAP 2d data...
Running K-Means on Raw data (This may take a while)...


/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


Running K-Means on UMAP 3d data...
Running K-Means on UMAP optimal (30d) data...
Running Hierarchical Clustering (K=2) on UMAP 2d data...
Running Hierarchical Clustering (K=2) on UMAP 3d data...
Running Hierarchical Clustering (K=2) on UMAP optimal (30d) data...
Running Hierarchical Clustering (K=2) on Raw data...


Running GMM (n_components=2) on UMAP 2d data...
Running GMM (n_components=2) on UMAP 3d data...
Running GMM (n_components=2) on UMAP optimal (30d) data...


/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-

In [ ]:
 #4. Compile the new features into a DataFrame
print("Compiling results...")
cluster_results = pd.DataFrame({
    'sample': meth_samples,
    'kmeans_umap_2_k2': kmeans_umap_2d_labels_k2,
    'kmeans_umap_3_k2': kmeans_umap_3d_labels_k2,
    'kmeans_umap_opt_k2': kmeans_umap_opt_labels_k2,  #ClaudeCode Ver. 1: was computed above but missing from this compile step
    'kmeans_raw_k2': kmeans_raw_labels_k2,
    'hierarchical_umap_2_k2': hierarchical_umap_2d_labels_k2,
    'hierarchical_umap_3_k2': hierarchical_umap_3d_labels_k2,
    'hierarchical_umap_opt_k2': hierarchical_umap_opt_labels_k2,
    'hierarchical_raw_k2': hierarchical_raw_labels_k2,
    'gmm_umap_2_k2': gmm_umap_2d_labels_k2,
    'gmm_umap_3_k2': gmm_umap_3d_labels_k2,
    'gmm_umap_opt_k2': gmm_umap_opt_labels_k2
})

# 5. Merge with the clinical dataset
print("Merging with clinical data...")
# Merging on 'sample' column. Using an inner or left join to ensure alignment
final_df = pd.merge(clin_df, cluster_results, on='sample', how='left')

# 6. Save the new dataset
output_file = '../data/processed/clinical_kirc_train_clustered_umap.csv'

In [7]:
final_df

,Unnamed: 0,sample,tissue_type.samples,kmeans_umap_2_k2,kmeans_umap_3_k2,kmeans_raw_k2,hierarchical_umap_2_k2,hierarchical_umap_3_k2,hierarchical_umap_opt_k2,hierarchical_raw_k2,gmm_umap_2_k2,gmm_umap_3_k2,gmm_umap_opt_k2
0,5,TCGA-DV-5573-01A,Tumor,0,0,1,0,0,0,0,0,0,1
1,8,TCGA-BP-4795-01A,Tumor,0,0,1,0,0,0,1,0,0,1
2,9,TCGA-BP-4795-11A,Normal,0,0,1,0,0,0,1,0,0,1
3,26,TCGA-DV-5565-01A,Tumor,0,0,1,0,0,0,0,0,0,1
4,27,TCGA-CJ-4869-11A,Normal,1,1,1,1,1,1,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
280,940,TCGA-A3-3385-11A,Normal,0,0,1,0,0,0,1,0,0,1
281,941,TCGA-A3-3385-01A,Tumor,0,0,1,0,0,0,0,0,0,1
282,942,TCGA-B8-5552-01B,Tumor,0,0,1,0,0,0,0,0,0,1
283,946,TCGA-B0-4821-01A,Tumor,1,1,1,0,0,1,0,1,1,0


In [8]:
final_df.to_csv(output_file, index=False)
print(f"Process complete! Saved new dataset to {output_file}")

Process complete! Saved new dataset to ../data/processed/clinical_kirc_train_clustered_umap.csv


In [ ]:
import pandas as pd
from sklearn.metrics import adjusted_rand_score

# 1. Load the dataset (if you are running this in a new script)
# final_df = pd.read_csv('clinical_kirc_all_sb_clustered.csv')

# 2. Define columns and drop ALL rows containing NaNs in these specific columns
cluster_columns = [
    'kmeans_umap_2_k2',
    'kmeans_umap_3_k2',
    'kmeans_umap_opt_k2',  #ClaudeCode Ver. 1: was missing from this list despite being computed above
    'kmeans_raw_k2',
    'hierarchical_umap_2_k2',
    'hierarchical_umap_3_k2',
    'hierarchical_umap_opt_k2',
    'hierarchical_raw_k2',
    'gmm_umap_2_k2',
    'gmm_umap_3_k2',
    'gmm_umap_opt_k2'
]
target_column = 'tissue_type.samples'  # Change this to the actual name of your ground truth column

columns_to_check = [target_column] + cluster_columns
valid_df = final_df.dropna(subset=columns_to_check).copy()

# Extract the true labels
true_labels = valid_df[target_column]

# 3. Compute and print the ARI for each feature
print(f"Adjusted Rand Index (ARI) Scores (Evaluated on {len(valid_df)} samples):")
print("-" * 55)

for col in cluster_columns:
    pred_labels = valid_df[col]
    ari_score = adjusted_rand_score(true_labels, pred_labels)
    print(f"{col:<25} : {ari_score:.4f}")